# 16 Time Shift

## Your Objective

Your dataset contains 22,000 support tickets from 700 users spread across 20 countries, each timestamped in UTC.

Your task is to convert each ticket to its user's local time and then calculate the ticket volume by local hour of day (0–23) across the full dataset.

See sample logic below:

![image](../images/fg2XhtqyKAuOCoRlhnLarXjnmw.avif)

In [1]:
import pandas as pd
import numpy as np

In [2]:
tickets = pd.read_csv('data/tickets.csv')
users = pd.read_csv('data/users.csv')

df = tickets.merge(users, how='left', on='user_id')
df.head()

,ticket_id,user_id,submitted_at_utc,country,timezone
0,1,546,2026-03-03T05:16:33,Pakistan,"GMT +05:00 — Islamabad, Karachi"
1,2,359,2026-03-06T12:26:33,Ghana,GMT +00:00 — Accra
2,3,655,2026-03-02T08:05:55,Saudi Arabia,"GMT +03:00 — Kuwait, Riyadh"
3,4,304,2026-04-02T04:33:34,Kenya,GMT +03:00 — Nairobi
4,5,98,2026-04-03T01:54:55,Pakistan,"GMT +05:00 — Islamabad, Karachi"


In [5]:

offset = df["timezone"].str.extract(r'([+-]\d{2}):(\d{2})')

minutes = offset[0].astype(int) * 60 + offset[1].astype(int)

minutes = np.where(
    offset[0] == "-",
    -minutes,
    minutes
)



In [6]:
df["submitted_at_utc"] = pd.to_datetime(
    df["submitted_at_utc"],
    utc=True
)

In [7]:
df["submitted_at_local"] = (
    df["submitted_at_utc"]
    + pd.to_timedelta(minutes, unit="m")
)

In [8]:
df["local_hour"] = df["submitted_at_local"].dt.hour

In [9]:
ticket_volume = (
    df.groupby("local_hour")
      .size()
      .reindex(range(24), fill_value=0)
)

In [10]:
print(ticket_volume)

local_hour
0        0
1        1
2       10
3       27
4       97
5      326
6      782
7     1587
8     2594
9     3587
10    3985
11    3572
12    2607
13    1606
14     749
15     328
16     114
17      22
18       3
19       1
20       1
21       1
22       0
23       0
dtype: int64
